# Day 58 - Feature Preprocessing
- Explain why ML input features need preprocessing.
- Scale numerical features with Keras preprocessing layers.
- Encode categorical features using lookup and one-hot techniques.
- Understand when categorical embeddings are preferable to one-hot vectors.
- Use Keras preprocessing layers such as `Normalization`, `StringLookup`, `IntegerLookup`, `CategoryEncoding`, `Hashing`, `TextVectorization`, and `Embedding`.
- Understand the purpose of **TF Transform (`tensorflow_transform`)** and why it helps prevent training/serving skew.
- Load datasets using **TensorFlow Datasets (`tensorflow_datasets`)**.
- Build a preprocessing pipeline that works with `tf.data`.
- Identify common preprocessing mistakes such as data leakage.
- Practice the concepts with exercises.

In [56]:
import sys
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

print("Python:", sys.version.split()[0])
print("TensorFlow:", tf.__version__)

np.random.seed(42)
tf.random.set_seed(42)


Python: 3.11.16
TensorFlow: 2.21.0


## 1. Why feature preprocessing matters

Neural networks generally work best when their inputs are represented numerically and have sensible scales.

Suppose a dataset contains:

- `age`: 18–80
- `income`: 20,000–2,000,000
- `city`: Bangalore / Mumbai / Delhi / ...
- `has_degree`: yes / no

These features have different types and scales.

A typical preprocessing strategy is:

| Feature | Raw type | Possible transformation |
|---|---|---|
| Age | numerical | normalization |
| Income | numerical | normalization or log transform |
| City | categorical string | integer lookup + one-hot/embedding |
| Has degree | categorical | binary encoding |
| Text | string | tokenization/vectorization |

The goal is to turn raw inputs into a representation that the model can learn from efficiently.


## 2. Numerical feature normalization

A common transformation is:

$$
z = \frac{x - \mu}{\sigma}
$$

where:

- $\mu$ = mean of the training data
- $\sigma$ = standard deviation of the training data

Keras provides the `Normalization` preprocessing layer.

The important workflow is:

```text
Training data
     ↓
normalizer.adapt(training_data)
     ↓
learn mean and variance
     ↓
normalizer(x)
```

**Never calculate preprocessing statistics using validation/test data.** Doing so can introduce data leakage.


In [57]:
# Example numerical data
ages = np.array([[20], [25], [30], [35], [40], [50], [60]], dtype=np.float32)

normalizer = keras.layers.Normalization()
normalizer.adapt(ages)

normalized_ages = normalizer(ages)

print("Learned mean:", normalizer.mean.numpy())
print("Learned variance:", normalizer.variance.numpy())
print("Original ages:")
print(ages.ravel())
print("Normalized ages:")
print(normalized_ages.numpy().round(3).ravel())


Learned mean: [[37.142857]]
Learned variance: [[170.40816]]
Original ages:
[20. 25. 30. 35. 40. 50. 60.]
Normalized ages:
[-1.313 -0.93  -0.547 -0.164  0.219  0.985  1.751]


### Why `adapt()` matters

`adapt()` is similar to fitting a preprocessing object.

For example:

```python
normalizer = keras.layers.Normalization()
normalizer.adapt(training_data)
```

The layer learns the statistics it needs from **training data**.

Later, the same layer can transform:

- training data
- validation data
- test data
- production/inference data

using the same learned statistics.


## 3. Normalizing multiple numerical features

A preprocessing layer can learn statistics for multiple columns/features.



In [58]:
# Each row = [age, income, years_experience]
X_train = np.array([
    [22, 30000, 1],
    [25, 45000, 2],
    [30, 60000, 5],
    [35, 85000, 8],
    [40, 100000, 12],
    [50, 140000, 20],
], dtype=np.float32)

normalizer = keras.layers.Normalization()
normalizer.adapt(X_train)

X_scaled = normalizer(X_train)

print("Original:")
print(X_train)
print("\nNormalized:")
print(np.round(X_scaled.numpy(), 2))


Original:
[[2.2e+01 3.0e+04 1.0e+00]
 [2.5e+01 4.5e+04 2.0e+00]
 [3.0e+01 6.0e+04 5.0e+00]
 [3.5e+01 8.5e+04 8.0e+00]
 [4.0e+01 1.0e+05 1.2e+01]
 [5.0e+01 1.4e+05 2.0e+01]]

Normalized:
[[-1.24 -1.27 -1.08]
 [-0.92 -0.86 -0.92]
 [-0.39 -0.45 -0.46]
 [ 0.14  0.23  0.  ]
 [ 0.67  0.64  0.61]
 [ 1.73  1.73  1.84]]


## 4. Categorical features

Categorical data represents discrete choices.

Example:

```text
city
-----
Bangalore
Mumbai
Delhi
Bangalore
Chennai
```

A neural network cannot directly consume arbitrary strings as useful numeric features.

One common pipeline is:

```text
String
  ↓
StringLookup
  ↓
Integer ID
  ↓
CategoryEncoding / Embedding
  ↓
Numeric representation
```


In [59]:
cities = np.array([
    ["Bangalore"],
    ["Mumbai"],
    ["Delhi"],
    ["Bangalore"],
    ["Chennai"],
    ["Mumbai"]
])

city_lookup = keras.layers.StringLookup(
    output_mode="int"
)

city_lookup.adapt(cities)

city_ids = city_lookup(cities)

print("Vocabulary:")
print(city_lookup.get_vocabulary())

print("\nInteger IDs:")
print(city_ids.numpy().ravel())


Vocabulary:
['[UNK]', np.str_('Mumbai'), np.str_('Bangalore'), np.str_('Delhi'), np.str_('Chennai')]

Integer IDs:
[2 1 3 2 4 1]


## 5. StringLookup and one-hot encoding

For small categorical vocabularies, one-hot encoding is often useful.

`StringLookup(output_mode="one_hot")` can combine vocabulary lookup and one-hot encoding.


In [60]:
city_one_hot = keras.layers.StringLookup(
    output_mode="one_hot"
)

city_one_hot.adapt(cities)

encoded = city_one_hot(cities)

print("Vocabulary:")
print(city_one_hot.get_vocabulary())

print("\nOne-hot encoded cities:")
print(encoded.numpy())


Vocabulary:
['[UNK]', np.str_('Mumbai'), np.str_('Bangalore'), np.str_('Delhi'), np.str_('Chennai')]

One-hot encoded cities:
[[0 0 1 0 0]
 [0 1 0 0 0]
 [0 0 0 1 0]
 [0 0 1 0 0]
 [0 0 0 0 1]
 [0 1 0 0 0]]


### Important detail: unknown categories

Real-world inference data can contain a category that was not present during training.

For example:

```text
Training: Bangalore, Mumbai, Delhi
Inference: Chennai
```

A lookup layer needs an **out-of-vocabulary (OOV)** strategy.

Keras lookup layers provide OOV handling so unseen categories do not necessarily crash the model.


In [61]:
training_cities = np.array([
    ["Bangalore"],
    ["Mumbai"],
    ["Delhi"],
])

lookup = keras.layers.StringLookup(
    output_mode="int",
    num_oov_indices=1
)
lookup.adapt(training_cities)

new_cities = np.array([
    ["Bangalore"],
    ["Chennai"],   # unseen
    ["Delhi"]
])

print("Vocabulary:", lookup.get_vocabulary())
print("Encoded:", lookup(new_cities).numpy().ravel())


Vocabulary: ['[UNK]', np.str_('Mumbai'), np.str_('Delhi'), np.str_('Bangalore')]
Encoded: [3 0 2]


## 6. IntegerLookup

Not every categorical feature arrives as a string.

For example:

```text
1 = Bronze
2 = Silver
3 = Gold
```

`IntegerLookup` is designed for integer-valued categorical features.


In [62]:
membership = np.array([
    [1],
    [2],
    [3],
    [2],
    [1],
], dtype=np.int32)

membership_lookup = keras.layers.IntegerLookup(
    output_mode="one_hot"
)

membership_lookup.adapt(membership)

print("Vocabulary:", membership_lookup.get_vocabulary())
print("Encoded:")
print(membership_lookup(membership).numpy())


Vocabulary: [-1, np.int64(2), np.int64(1), np.int64(3)]
Encoded:
[[0 0 1 0]
 [0 1 0 0]
 [0 0 0 1]
 [0 1 0 0]
 [0 0 1 0]]


## 7. CategoryEncoding

`CategoryEncoding` converts integer IDs into representations such as one-hot or multi-hot vectors.

This is useful when the integer IDs have already been created.



In [63]:
category_ids = tf.constant([0, 1, 2, 1, 0])

encoder = keras.layers.CategoryEncoding(
    num_tokens=3,
    output_mode="one_hot"
)

encoded = encoder(category_ids)

print(encoded.numpy())


[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]
 [0. 1. 0.]
 [1. 0. 0.]]


## 8. One-hot encoding vs embeddings

### One-hot encoding

Suppose there are 10,000 possible product categories.

A category might become:

```text
[0, 0, 0, ... 1 ... 0]
```

This vector has 10,000 dimensions.

That can become inefficient for high-cardinality features.

### Embedding

Instead, an embedding maps each category to a dense learned vector:

```text
category ID
    ↓
Embedding
    ↓
[0.21, -0.48, 0.77, ...]
```

For example, 10,000 categories could use a 16- or 32-dimensional embedding.

The embedding vectors are learned during model training.


In [64]:
# A small embedding example
num_categories = 6
embedding_dim = 3

embedding = keras.layers.Embedding(
    input_dim=num_categories,
    output_dim=embedding_dim
)

category_ids = tf.constant([[0], [1], [2], [3], [4], [5]])

embedded = embedding(category_ids)

print("Shape:", embedded.shape)
print("\nEmbedding vectors:")
print(embedded.numpy().round(3))


Shape: (6, 1, 3)

Embedding vectors:
[[[ 0.002 -0.005 -0.01 ]]

 [[ 0.039  0.049  0.016]]

 [[ 0.039 -0.016 -0.017]]

 [[ 0.035  0.046  0.024]]

 [[-0.045  0.038 -0.009]]

 [[ 0.05   0.046 -0.033]]]


## 9. A small categorical embedding model

Here we build a simple model that receives an integer category ID and predicts a binary target.

The important part is the embedding layer.



In [65]:
category_input = keras.Input(shape=(1,), dtype=tf.int32, name="category")

x = keras.layers.Embedding(
    input_dim=10,
    output_dim=4
)(category_input)

x = keras.layers.Flatten()(x)
x = keras.layers.Dense(8, activation="relu")(x)
output = keras.layers.Dense(1, activation="sigmoid")(x)

embedding_model = keras.Model(category_input, output)

embedding_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

embedding_model.summary()


Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ category (InputLayer)           │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_5 (Embedding)         │ (None, 1, 4)           │            40 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 4)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 8)              │            40 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 89 (356.00 B)

 Trainable params: 89 (356.00 B)

 Non-trainable params: 0 (0.00 B)

## 10. Hashing for high-cardinality categories

Sometimes the vocabulary is huge or changes frequently.

Instead of explicitly maintaining every category, hashing can map values into a fixed number of buckets.

Conceptually:

```text
raw category
     ↓
hash function
     ↓
bucket ID
```

This keeps the representation size bounded.

The trade-off is that different categories can occasionally map to the same bucket (**hash collision**).


In [66]:
values = tf.constant([
    ["Bangalore"],
    ["Mumbai"],
    ["Delhi"],
    ["Chennai"],
    ["Bangalore"]
])

hasher = keras.layers.Hashing(num_bins=8)

hashed = hasher(values)

print("Hashed bucket IDs:")
print(hashed.numpy().ravel())


Hashed bucket IDs:
[2 1 0 3 2]


## 11. TextVectorization

Text is another type of raw feature.

`TextVectorization` can perform several steps:

```text
raw text
   ↓
standardization
   ↓
tokenization
   ↓
vocabulary lookup
   ↓
integer sequences / counts / TF-IDF
```



In [67]:
texts = np.array([
    ["TensorFlow is useful for deep learning"],
    ["Keras makes model development easier"],
    ["TensorFlow provides the data API"],
    ["Deep learning models need good data"]
])

text_vectorizer = keras.layers.TextVectorization(
    max_tokens=50,
    output_mode="int",
    output_sequence_length=8
)

text_vectorizer.adapt(texts)

print("Vocabulary:")
print(text_vectorizer.get_vocabulary()[:20])

print("\nVectorized text:")
print(text_vectorizer(texts).numpy())


Vocabulary:
['', '[UNK]', np.str_('tensorflow'), np.str_('learning'), np.str_('deep'), np.str_('data'), np.str_('useful'), np.str_('the'), np.str_('provides'), np.str_('need'), np.str_('models'), np.str_('model'), np.str_('makes'), np.str_('keras'), np.str_('is'), np.str_('good'), np.str_('for'), np.str_('easier'), np.str_('development'), np.str_('api')]

Vectorized text:
[[ 2 14  6 16  4  3  0  0]
 [13 12 11 18 17  0  0  0]
 [ 2  8  7  5 19  0  0  0]
 [ 4  3 10  9 15  5  0  0]]


## 12. TextVectorization with TF-IDF

For some classical text problems, TF-IDF is useful.

`TextVectorization(output_mode="tf_idf")` creates a fixed-size numerical representation based on word importance.


In [68]:
tfidf_vectorizer = keras.layers.TextVectorization(
    max_tokens=50,
    output_mode="tf_idf"
)

tfidf_vectorizer.adapt(texts)

tfidf_features = tfidf_vectorizer(texts)

print("TF-IDF feature shape:", tfidf_features.shape)
print(tfidf_features.numpy().round(3))


TF-IDF feature shape: (4, 19)
[[0.    0.847 0.847 0.847 0.    1.099 0.    0.    0.    0.    0.    0.
  0.    1.099 0.    1.099 0.    0.    0.   ]
 [0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    1.099 1.099
  1.099 0.    0.    0.    1.099 1.099 0.   ]
 [0.    0.847 0.    0.    0.847 0.    1.099 1.099 0.    0.    0.    0.
  0.    0.    0.    0.    0.    0.    1.099]
 [0.    0.    0.847 0.847 0.847 0.    0.    0.    1.099 1.099 0.    0.
  0.    0.    1.099 0.    0.    0.    0.   ]]


## 13. Combining preprocessing with a Keras model

A powerful design is to keep preprocessing inside the model.

For example:

```text
raw age + raw city
       ↓
Normalization + StringLookup
       ↓
encoded numerical features
       ↓
Dense layers
       ↓
prediction
```

This can make the saved model easier to use consistently.


In [69]:
# Example: raw age + raw city
age_input = keras.Input(shape=(1,), dtype=tf.float32, name="age")
city_input = keras.Input(shape=(1,), dtype=tf.string, name="city")

age_normalizer = keras.layers.Normalization()
age_normalizer.adapt(X_train[:, :1])

city_encoder = keras.layers.StringLookup(
    output_mode="one_hot"
)
city_encoder.adapt(cities)

age_features = age_normalizer(age_input)
city_features = city_encoder(city_input)

features = keras.layers.Concatenate()([
    age_features,
    city_features
])

x = keras.layers.Dense(16, activation="relu")(features)
x = keras.layers.Dense(8, activation="relu")(x)
output = keras.layers.Dense(1, activation="sigmoid")(x)

preprocessing_model = keras.Model(
    inputs={"age": age_input, "city": city_input},
    outputs=output
)

preprocessing_model.summary()


Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ age (InputLayer)    │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ city (InputLayer)   │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization_17    │ (None, 1)         │          3 │ age[0][0]         │
│ (Normalization)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ string_lookup_23    │ (None, 5)         │          0 │ city[0][0]        │
│ (StringLookup)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_5       │ (None, 6)         │          0 │ normalization_17… │
│ (Concatenate)       │                   │            │ string_lookup_23… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_20 (Dense)    │ (None, 16)        │        112 │ concatenate_5[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_21 (Dense)    │ (None, 8)         │        136 │ dense_20[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_22 (Dense)    │ (None, 1)         │          9 │ dense_21[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 260 (1.02 KB)

 Trainable params: 257 (1.00 KB)

 Non-trainable params: 3 (16.00 B)

## 14. Testing the combined model

The model can now accept raw values instead of requiring us to manually preprocess them first.


In [70]:
sample = {
    "age": np.array([[28], [45]], dtype=np.float32),
    "city": np.array([["Bangalore"], ["Mumbai"]])
}

predictions = preprocessing_model(sample)

print("Predictions:")
print(predictions.numpy().ravel())


Predictions:
[0.55959284 0.6177256 ]


## 15. Data leakage — a critical concept

Suppose you normalize the entire dataset before splitting it:

```text
ALL DATA
   ↓
calculate mean/std
   ↓
normalize
   ↓
train/test split
```

The test set has influenced the preprocessing statistics.

That is leakage.

Prefer:

```text
ALL DATA
   ↓
train/test split
   ↓
training data → learn statistics
                    ↓
          same transformation
                    ↓
          train + validation + test
```

The test set should remain unseen until evaluation.


## 16. Preprocessing with tf.data

The preprocessing layer can be used together with the TensorFlow Data API.

This is useful when building scalable input pipelines.



In [71]:
raw_numbers = tf.constant([
    [10.0],
    [20.0],
    [30.0],
    [40.0],
    [50.0],
    [60.0]
])

dataset = tf.data.Dataset.from_tensor_slices(raw_numbers)
dataset = dataset.batch(2)

number_normalizer = keras.layers.Normalization()
number_normalizer.adapt(raw_numbers)

normalized_dataset = dataset.map(
    lambda x: number_normalizer(x)
)

for batch in normalized_dataset:
    print(batch.numpy().round(2))


[[-1.46]
 [-0.88]]
[[-0.29]
 [ 0.29]]
[[0.88]
 [1.46]]


## 17. TF Transform — the problem it solves

**TF Transform (`tensorflow_transform`, commonly imported as `tft`)** is designed for preprocessing that needs to be computed from data and then applied consistently.

Imagine that training data has:

```text
income mean = 75,000
income std  = 25,000
```

You want the same transformation to happen when the model receives production data.

TF Transform can:

1. Analyze training data.
2. Compute preprocessing statistics.
3. Build a preprocessing graph.
4. Apply that transformation consistently.
5. Export the preprocessing logic for serving.

This helps address **training-serving skew**.

### Typical TF Transform architecture

```text
Raw training data
       ↓
     tft
       ↓
Analyze + transform
       ↓
Transformed training data
       ↓
      Model

Production raw data
       ↓
Saved preprocessing graph
       ↓
Same transformation
       ↓
     Model
```

> **Environment note:** `tensorflow_transform` has its own compatibility requirements and is not always installed with TensorFlow. The next cell checks whether it is available rather than failing the entire notebook.


In [72]:
# Optional TF Transform availability check
try:
    import tensorflow_transform as tft
    print("tensorflow_transform is installed.")
    print("Version:", getattr(tft, "__version__", "version unavailable"))
except ImportError:
    print("tensorflow_transform is not installed in this environment.")
    print("The conceptual workflow above is still important.")
    print("If needed, install a TF Transform version compatible with your TensorFlow environment.")


tensorflow_transform is not installed in this environment.
The conceptual workflow above is still important.
If needed, install a TF Transform version compatible with your TensorFlow environment.


## 18. TF Transform example pattern

A typical TF Transform preprocessing function looks conceptually like this:

```python
def preprocessing_fn(inputs):
    x = inputs["income"]
    x_scaled = tft.scale_to_z_score(x)

    return {
        "income_scaled": x_scaled
    }
```

`analyze_and_transform_dataset()` then uses the training data to analyze the required statistics and produces transformed data plus a reusable transformation graph.

A key benefit is that statistics such as mean and variance are computed from the appropriate analysis data rather than being accidentally recomputed independently at serving time.


In [73]:
# Illustrative TF Transform function.
# This cell only runs if tensorflow_transform is installed.

try:
    import tensorflow_transform as tft

    def preprocessing_fn(inputs):
        income = inputs["income"]
        age = inputs["age"]

        return {
            "income_scaled": tft.scale_to_z_score(income),
            "age_scaled": tft.scale_to_z_score(age),
        }

    print("TF Transform preprocessing_fn defined successfully.")
except ImportError:
    print("Skipping TF Transform code because tensorflow_transform is not installed.")


Skipping TF Transform code because tensorflow_transform is not installed.


## 19. TensorFlow Datasets (TFDS)

**TensorFlow Datasets** provides many ready-to-use datasets.

The common pattern is:

```python
import tensorflow_datasets as tfds

ds = tfds.load(
    "dataset_name",
    split="train",
    as_supervised=True
)
```

TFDS handles dataset downloading, preparation, and standardized dataset access.

Examples include datasets for:

- images
- text
- audio
- structured data
- classification
- question answering



In [74]:
# Optional TFDS availability check
try:
    import tensorflow_datasets as tfds
    print("tensorflow_datasets is installed.")
    print("Version:", getattr(tfds, "__version__", "version unavailable"))
except ImportError:
    print("tensorflow_datasets is not installed.")


tensorflow_datasets is not installed.


## 20. Loading MNIST using TFDS

If TFDS is installed, we can load MNIST.

The first execution may download dataset files, so it requires an internet connection.



In [75]:
try:
    import tensorflow_datasets as tfds

    (train_ds, test_ds), info = tfds.load(
        "mnist",
        split=["train", "test"],
        as_supervised=True,
        with_info=True
    )

    print("Dataset info:")
    print(info)
    print("\nTrain examples:", info.splits["train"].num_examples)
    print("Test examples:", info.splits["test"].num_examples)

except ImportError:
    print("Install tensorflow-datasets to run this example.")
except Exception as e:
    print("TFDS example could not run in this environment.")
    print("Reason:", type(e).__name__, str(e)[:300])


Install tensorflow-datasets to run this example.


## 21. Inspecting a TFDS dataset

A `tf.data.Dataset` can be inspected with operations such as:

- `take()`
- `batch()`
- `shuffle()`
- `map()`
- `prefetch()`



In [76]:
try:
    for image, label in train_ds.take(1):
        print("Image shape:", image.shape)
        print("Image dtype:", image.dtype)
        print("Label:", label.numpy())
except NameError:
    print("Run the previous TFDS loading cell first.")
except Exception as e:
    print("Could not inspect the dataset:", e)


Run the previous TFDS loading cell first.


## 22. A complete mini preprocessing pipeline

Let's combine several ideas.

Dataset:

```text
age
income
city
```

Pipeline:

```text
age    → Normalization
income → Normalization
city   → StringLookup → one-hot
                 ↓
          concatenate
                 ↓
             Dense NN
```


In [77]:
# Small synthetic dataset
data = pd.DataFrame({
    "age": [22, 25, 28, 35, 42, 50, 31, 39],
    "income": [30000, 42000, 50000, 72000, 90000, 120000, 65000, 81000],
    "city": [
        "Bangalore", "Mumbai", "Delhi", "Bangalore",
        "Chennai", "Mumbai", "Delhi", "Bangalore"
    ],
    "target": [0, 0, 1, 0, 1, 1, 1, 1]
})

display(data)


,age,income,city,target
0,22,30000,Bangalore,0
1,25,42000,Mumbai,0
2,28,50000,Delhi,1
3,35,72000,Bangalore,0
4,42,90000,Chennai,1
5,50,120000,Mumbai,1
6,31,65000,Delhi,1
7,39,81000,Bangalore,1


In [78]:
# Prepare feature arrays
# age_data = data[["age"]].astype("float32").values
# income_data = data[["income"]].astype("float32").values
# city_data = data[["city"]].astype("str").values
# y_data = data["target"].astype("float32").values

# Prepare feature arrays
age_data = data[["age"]].to_numpy(dtype=np.float32)
income_data = data[["income"]].to_numpy(dtype=np.float32)

# Explicitly create a TensorFlow string tensor
city_data = tf.convert_to_tensor(
    data[["city"]].to_numpy(dtype=str),
    dtype=tf.string
)

y_data = data["target"].to_numpy(dtype=np.float32)

print("age:", age_data.shape, age_data.dtype)
print("income:", income_data.shape, income_data.dtype)
print("city:", city_data.shape, city_data.dtype)
print("target:", y_data.shape, y_data.dtype)

age_norm = keras.layers.Normalization()
age_norm.adapt(age_data)

income_norm = keras.layers.Normalization()
income_norm.adapt(income_data)

city_lookup = keras.layers.StringLookup(output_mode="one_hot")
city_lookup.adapt(city_data)

age_input = keras.Input(shape=(1,), dtype=tf.float32, name="age")
income_input = keras.Input(shape=(1,), dtype=tf.float32, name="income")
city_input = keras.Input(shape=(1,), dtype=tf.string, name="city")

age_x = age_norm(age_input)
income_x = income_norm(income_input)
city_x = city_lookup(city_input)

x = keras.layers.Concatenate()([age_x, income_x, city_x])
x = keras.layers.Dense(16, activation="relu")(x)
x = keras.layers.Dense(8, activation="relu")(x)
output = keras.layers.Dense(1, activation="sigmoid")(x)

mini_model = keras.Model(
    inputs={
        "age": age_input,
        "income": income_input,
        "city": city_input
    },
    outputs=output
)

mini_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

mini_model.summary()


age: (8, 1) float32
income: (8, 1) float32
city: (8, 1) <dtype: 'string'>
target: (8,) float32


Model: "functional_9"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ age (InputLayer)    │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ income (InputLayer) │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ city (InputLayer)   │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization_19    │ (None, 1)         │          3 │ age[0][0]         │
│ (Normalization)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization_20    │ (None, 1)         │          3 │ income[0][0]      │
│ (Normalization)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ string_lookup_24    │ (None, 5)         │          0 │ city[0][0]        │
│ (StringLookup)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_6       │ (None, 7)         │          0 │ normalization_19… │
│ (Concatenate)       │                   │            │ normalization_20… │
│                     │                   │            │ string_lookup_24… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_23 (Dense)    │ (None, 16)        │        128 │ concatenate_6[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_24 (Dense)    │ (None, 8)         │        136 │ dense_23[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_25 (Dense)    │ (None, 1)         │          9 │ dense_24[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 279 (1.10 KB)

 Trainable params: 273 (1.07 KB)

 Non-trainable params: 6 (32.00 B)

In [79]:
history = mini_model.fit(
    {
        "age": age_data,
        "income": income_data,
        "city": city_data
    },
    y_data,
    epochs=10,
    verbose=0
)

print("Final training loss:", history.history["loss"][-1])
print("Final training accuracy:", history.history["accuracy"][-1])


Final training loss: 0.6996737718582153
Final training accuracy: 0.75


## 23. Preprocessing checklist

Before training a model, ask:

### Numerical features
- Do I need normalization?
- Are there extreme outliers?
- Are missing values handled?

### Categorical features
- Is the vocabulary small enough for one-hot encoding?
- Would an embedding be more efficient?
- How are unknown categories handled?

### Text
- Do I need tokenization?
- Should I use integer sequences, counts, or TF-IDF?
- Is vocabulary learned only from training data?

### Pipeline
- Are preprocessing statistics learned only from training data?
- Is preprocessing identical during training and inference?
- Can preprocessing be included in the model?
- Would TF Transform be useful for large/production pipelines?

### Dataset
- Can `tf.data` pipeline the data efficiently?
- Can TFDS provide a standardized dataset?


# Exercises

Try these yourself before looking at the solutions.

---

## Exercise 1 — Normalize a feature

Create a `Normalization` layer for:

```python
temperatures = [[20], [25], [30], [35], [40]]
```

1. Call `adapt()`.
2. Transform the values.
3. Print the learned mean and variance.

---

## Exercise 2 — One-hot encode categories

Given:

```python
colors = [["red"], ["blue"], ["green"], ["red"], ["blue"]]
```

Create a `StringLookup` layer that produces one-hot vectors.

---

## Exercise 3 — Handle an unknown category

Train a lookup layer using:

```text
Bangalore
Mumbai
Delhi
```

Then encode:

```text
Bangalore
Chennai
Delhi
```

Identify which value corresponds to the OOV category.

---

## Exercise 4 — Embedding dimensions

Suppose you have 50,000 product categories.

Compare:

- one-hot representation
- 8-dimensional embedding
- 32-dimensional embedding

Write a short explanation of why embeddings can be much more memory-efficient.

---

## Exercise 5 — Text preprocessing

Create a `TextVectorization` layer for:

```text
"machine learning is useful"
"deep learning is powerful"
"tensorflow supports machine learning"
```

Return integer sequences with a maximum sequence length of 6.

---

## Exercise 6 — Data leakage

Explain why this is problematic:

```python
normalizer.adapt(all_data)

train_data = all_data[:800]
test_data = all_data[800:]
```

Rewrite the logic so that test information does not influence the normalization statistics.

---

## Exercise 7 — Build a preprocessing model

Create a model with:

- `age` → normalization
- `city` → one-hot encoding
- concatenate the features
- Dense(16, ReLU)
- Dense(1, sigmoid)

Use synthetic data and train for at least 5 epochs.

---

## Exercise 8 — TFDS exploration

If TFDS is available:

1. Load `mnist`.
2. Print the number of training examples.
3. Inspect one sample.
4. Print its image shape and label.
5. Create a batched, prefetched dataset.



# Exercise Solutions

Try each exercise independently first.


In [80]:
# Exercise 1 solution
temperatures = np.array([[20], [25], [30], [35], [40]], dtype=np.float32)

temp_normalizer = keras.layers.Normalization()
temp_normalizer.adapt(temperatures)

print("Mean:", temp_normalizer.mean.numpy())
print("Variance:", temp_normalizer.variance.numpy())
print("Normalized:")
print(temp_normalizer(temperatures).numpy().round(3))


Mean: [[30.]]
Variance: [[50.]]
Normalized:
[[-1.414]
 [-0.707]
 [ 0.   ]
 [ 0.707]
 [ 1.414]]


In [81]:
# Exercise 2 solution
colors = np.array([
    ["red"],
    ["blue"],
    ["green"],
    ["red"],
    ["blue"]
])

color_encoder = keras.layers.StringLookup(output_mode="one_hot")
color_encoder.adapt(colors)

print("Vocabulary:", color_encoder.get_vocabulary())
print("One-hot:")
print(color_encoder(colors).numpy())


Vocabulary: ['[UNK]', np.str_('red'), np.str_('blue'), np.str_('green')]
One-hot:
[[0 1 0 0]
 [0 0 1 0]
 [0 0 0 1]
 [0 1 0 0]
 [0 0 1 0]]


In [82]:
# Exercise 3 solution
training_cities = np.array([
    ["Bangalore"],
    ["Mumbai"],
    ["Delhi"]
])

city_lookup_ex3 = keras.layers.StringLookup(
    output_mode="int",
    num_oov_indices=1
)
city_lookup_ex3.adapt(training_cities)

test_cities = np.array([
    ["Bangalore"],
    ["Chennai"],
    ["Delhi"]
])

encoded = city_lookup_ex3(test_cities)

print("Vocabulary:", city_lookup_ex3.get_vocabulary())
print("Encoded:", encoded.numpy().ravel())
print("Chennai is mapped to the OOV index.")


Vocabulary: ['[UNK]', np.str_('Mumbai'), np.str_('Delhi'), np.str_('Bangalore')]
Encoded: [3 0 2]
Chennai is mapped to the OOV index.


In [83]:
# Exercise 4 solution
num_categories = 50_000

one_hot_dimensions = num_categories
embedding_8_dimensions = 8
embedding_32_dimensions = 32

print("One-hot dimensions per category:", one_hot_dimensions)
print("8D embedding dimensions:", embedding_8_dimensions)
print("32D embedding dimensions:", embedding_32_dimensions)

print("\nOne-hot requires a vector with 50,000 positions.")
print("An embedding can represent each category using a much smaller dense vector.")
print("The embedding vectors are learned rather than manually assigned.")


One-hot dimensions per category: 50000
8D embedding dimensions: 8
32D embedding dimensions: 32

One-hot requires a vector with 50,000 positions.
An embedding can represent each category using a much smaller dense vector.
The embedding vectors are learned rather than manually assigned.


In [84]:
# Exercise 5 solution
exercise_texts = np.array([
    ["machine learning is useful"],
    ["deep learning is powerful"],
    ["tensorflow supports machine learning"]
])

exercise_vectorizer = keras.layers.TextVectorization(
    max_tokens=50,
    output_mode="int",
    output_sequence_length=6
)

exercise_vectorizer.adapt(exercise_texts)

print("Vocabulary:", exercise_vectorizer.get_vocabulary())
print("\nSequences:")
print(exercise_vectorizer(exercise_texts).numpy())


Vocabulary: ['', '[UNK]', np.str_('learning'), np.str_('machine'), np.str_('is'), np.str_('useful'), np.str_('tensorflow'), np.str_('supports'), np.str_('powerful'), np.str_('deep')]

Sequences:
[[3 2 4 5 0 0]
 [9 2 4 8 0 0]
 [6 7 3 2 0 0]]


In [85]:
# Exercise 6 solution
all_data = np.array([
    [10.0], [20.0], [30.0], [40.0],
    [50.0], [60.0], [70.0], [80.0],
    [90.0], [100.0]
], dtype=np.float32)

train_data = all_data[:8]
test_data = all_data[8:]

# Correct: learn preprocessing statistics only from training data.
correct_normalizer = keras.layers.Normalization()
correct_normalizer.adapt(train_data)

print("Training mean:", correct_normalizer.mean.numpy())
print("Training variance:", correct_normalizer.variance.numpy())

print("\nNormalized training data:")
print(correct_normalizer(train_data).numpy().round(2))

print("\nNormalized test data using the SAME training statistics:")
print(correct_normalizer(test_data).numpy().round(2))


Training mean: [[45.]]
Training variance: [[525.]]

Normalized training data:
[[-1.53]
 [-1.09]
 [-0.65]
 [-0.22]
 [ 0.22]
 [ 0.65]
 [ 1.09]
 [ 1.53]]

Normalized test data using the SAME training statistics:
[[1.96]
 [2.4 ]]


In [86]:
# Exercise 7 — Corrected solution

exercise_df = pd.DataFrame({
    "age": [21, 25, 29, 34, 40, 46, 52, 58],
    "city": [
        "Bangalore", "Mumbai", "Delhi", "Bangalore",
        "Mumbai", "Delhi", "Bangalore", "Mumbai"
    ],
    "target": [0, 0, 1, 0, 1, 1, 1, 1]
})

# Numerical data
ages = exercise_df[["age"]].to_numpy(dtype=np.float32)

# IMPORTANT:
# Convert categorical data directly to a TensorFlow string tensor
cities = tf.convert_to_tensor(
    exercise_df[["city"]].to_numpy(dtype=str),
    dtype=tf.string
)

# Target
targets = exercise_df["target"].to_numpy(dtype=np.float32)


# --------------------------------------------------
# Preprocessing layers
# --------------------------------------------------

age_preprocessor = keras.layers.Normalization()
age_preprocessor.adapt(ages)

city_preprocessor = keras.layers.StringLookup(
    output_mode="one_hot"
)
city_preprocessor.adapt(cities)


# --------------------------------------------------
# Model inputs
# --------------------------------------------------

age_input = keras.Input(
    shape=(1,),
    dtype=tf.float32,
    name="age"
)

city_input = keras.Input(
    shape=(1,),
    dtype=tf.string,
    name="city"
)


# --------------------------------------------------
# Preprocessing
# --------------------------------------------------

age_features = age_preprocessor(age_input)
city_features = city_preprocessor(city_input)


# --------------------------------------------------
# Combine features
# --------------------------------------------------

features = keras.layers.Concatenate()([
    age_features,
    city_features
])


# --------------------------------------------------
# Neural network
# --------------------------------------------------

x = keras.layers.Dense(
    16,
    activation="relu"
)(features)

output = keras.layers.Dense(
    1,
    activation="sigmoid"
)(x)


# --------------------------------------------------
# Create model
# --------------------------------------------------

exercise_model = keras.Model(
    inputs={
        "age": age_input,
        "city": city_input
    },
    outputs=output
)


# --------------------------------------------------
# Compile
# --------------------------------------------------

exercise_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


# --------------------------------------------------
# Train
# --------------------------------------------------

history = exercise_model.fit(
    {
        "age": ages,
        "city": cities
    },
    targets,
    epochs=5,
    verbose=1
)

print("\nExercise 7 model trained successfully!")

Epoch 1/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 609ms/step - accuracy: 0.1250 - loss: 0.8811
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.1250 - loss: 0.8766
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.1250 - loss: 0.8721
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.1250 - loss: 0.8676
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.1250 - loss: 0.8633

Exercise 7 model trained successfully!


In [87]:
# Exercise 8 solution
try:
    import tensorflow_datasets as tfds

    ds, info = tfds.load(
        "mnist",
        split="train",
        with_info=True,
        as_supervised=True
    )

    print("Training examples:", info.splits["train"].num_examples)

    for image, label in ds.take(1):
        print("Image shape:", image.shape)
        print("Image dtype:", image.dtype)
        print("Label:", label.numpy())

    pipeline = (
        ds
        .shuffle(10_000)
        .batch(32)
        .prefetch(tf.data.AUTOTUNE)
    )

    print("Batched + prefetched TFDS pipeline created.")

except ImportError:
    print("tensorflow-datasets is not installed.")
except Exception as e:
    print("TFDS exercise could not run:", type(e).__name__, str(e)[:300])


tensorflow-datasets is not installed.


# Quick Revision Sheet

## Numerical preprocessing

```python
normalizer = keras.layers.Normalization()
normalizer.adapt(train_data)
x = normalizer(x)
```

**Remember:** `adapt()` should use training data.

---

## Categorical preprocessing

```python
lookup = keras.layers.StringLookup(output_mode="one_hot")
lookup.adapt(train_categories)
x = lookup(categories)
```

For integer categories:

```python
keras.layers.IntegerLookup(...)
```

---

## Embeddings

```python
embedding = keras.layers.Embedding(
    input_dim=num_categories,
    output_dim=embedding_dim
)
```

Use embeddings when categorical cardinality is large and a dense learned representation is useful.

---

## Text

```python
vectorizer = keras.layers.TextVectorization(...)
vectorizer.adapt(training_text)
```

Possible outputs include:

- integer sequences
- binary
- count
- TF-IDF

---

## Hashing

```python
keras.layers.Hashing(num_bins=...)
```

Useful for very large/dynamic categorical vocabularies, with possible hash collisions.

---

## TF Transform

Main purpose:

```text
Analyze training data
       ↓
Learn preprocessing statistics
       ↓
Build reusable transformation
       ↓
Use consistently in production
```

Main benefit:

**reduces training/serving skew and helps make preprocessing reproducible.**

---

## TFDS

```python
import tensorflow_datasets as tfds

ds = tfds.load(
    "mnist",
    split="train"
)
```

TFDS provides standardized access to many datasets.

---

# Mental model

```text
                    RAW DATA
                       │
          ┌────────────┼────────────┐
          ↓            ↓            ↓
     Numerical     Categorical     Text
          │            │            │
    Normalization   Lookup       Vectorization
          │            │            │
          │       One-hot /       Tokens /
          │       Embedding       TF-IDF
          └────────────┼────────────┘
                       ↓
                  tf.data
                       ↓
                 Keras Model
                       ↓
                  Prediction

Production:
       Training preprocessing
              ↓
       Same preprocessing
              ↓
          New data
```

### The most important Day 58 lesson

> **Preprocessing is part of the ML system, not just a data-cleaning step.**

If preprocessing differs between training and inference, even a well-trained model can perform badly.
